# Test pipeline – runs on GitHub Actions against Neon

Started by the workflow *Test pipeline* (`.github/workflows/test.yml`). Runs the real notebooks
`Ingest_data` and `Load_dwh` and checks the bucket and the DWH after each step. Any failed check fails the workflow.

| Test | Mode | What it proves | Data afterwards |
|---|---|---|---|
| **A. Incremental** | `incremental` only | newest deltadate removed from the bucket, DWH emptied → Load_dwh loads everything without it → Ingest fetches it again from the API, Load_dwh loads only that | same as before |
| **C. Idempotence** | both | pipeline run once to catch up, a second run changes nothing | unchanged (+ any new day) |
| **B. Upsert** | both | loading existing rows again updates them (no duplicates), a changed value arrives, `dim_city` keeps its key (SCD1) | unchanged – done in a transaction that is rolled back |
| **Checks** | both | `sql/checks/dwh_checks.sql`: every city every day, no gaps, no orphans, correct deltadate | – |

In [ ]:
# parameters (the workflow overrides MODE)
MODE = "upsert"         # "upsert": B + C, changes nothing | "incremental": A + B + C

## Setup

In [ ]:
import io
import re
import sys
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import papermill as pm
import psycopg
from botocore.config import Config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from config.settings import (DATABASE_URL, S3_ACCESS_KEY_ID, S3_BUCKET, S3_ENDPOINT_URL, S3_REGION,
                             S3_SECRET_ACCESS_KEY, STORAGE)

assert STORAGE == "s3", "the tests run against the Neon bucket (STORAGE=s3)"
assert MODE in ("upsert", "incremental"), MODE
s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT_URL, region_name=S3_REGION, aws_access_key_id=S3_ACCESS_KEY_ID,
                  aws_secret_access_key=S3_SECRET_ACCESS_KEY, config=Config(s3={"addressing_style": "path"}))
conn = psycopg.connect(DATABASE_URL, autocommit=True)
notices = []
conn.add_notice_handler(lambda d: notices.append(d.message_primary))
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
results = []
print(f"mode {MODE} | bucket {S3_BUCKET} | DWH {conn.info.host}/{conn.info.dbname}")

## Functions

In [ ]:
def list_keys(prefix):
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=prefix)
    return sorted(o["Key"] for page in pages for o in page.get("Contents", []) if o["Key"].endswith(".csv"))


def read_csv(key):
    return pd.read_csv(s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"])


def deltadate_of(key):
    return key.split("/")[1].split("=")[1]          # weather/deltadate=2026-10-06/Atlanta.csv -> 2026-10-06


def query(sql):
    cur = conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def value(sql):
    return conn.execute(sql).fetchone()[0]


def stage(table, df):
    columns = query(f"""SELECT column_name FROM information_schema.columns
                        WHERE table_schema = 'stg' AND table_name = '{table}' ORDER BY ordinal_position""")["column_name"]
    data = io.StringIO()
    df.reindex(columns=columns).to_csv(data, index=False, header=False)
    conn.execute(f"TRUNCATE stg.{table}")
    with conn.cursor().copy(f"COPY stg.{table} ({', '.join(columns)}) FROM STDIN WITH (FORMAT csv)") as copy:
        copy.write(data.getvalue())


def last_notice_rows():
    """Row count of the last procedure NOTICE, e.g. 'fact_weather_daily: 23 rows inserted or updated' -> 23"""
    return int(re.search(r": (\d+) ", notices[-1]).group(1))


def run(notebook, step):
    """Runs a pipeline notebook like the daily workflow does; the executed copy goes to output/."""
    pm.execute_notebook(str(ROOT / "notebooks" / f"{notebook}.ipynb"), str(OUT / f"{step}_{notebook}.ipynb"),
                        cwd=str(ROOT / "notebooks"), progress_bar=False)
    print(f"ran {notebook} ({step})")


def state():
    return query("""SELECT (SELECT count(*) FROM dwh.fact_weather_daily) AS fact_rows,
                           (SELECT count(*) FROM dwh.dim_date) AS dates,
                           (SELECT count(*) FROM dwh.dim_city) AS cities,
                           (SELECT max(deltadate)::text FROM dwh.fact_weather_daily) AS max_deltadate""").iloc[0].to_dict()


def fact_value(city, deltadate):
    return value(f"""SELECT f.temperature_2m_max FROM dwh.fact_weather_daily f JOIN dwh.dim_city c USING (city_key)
                     WHERE c.city = '{city}' AND f.deltadate = '{deltadate}' ORDER BY f.date_key DESC LIMIT 1""")


def same_values(a, b):
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return False
    num = a.select_dtypes("number").columns
    return bool(np.allclose(a[num], b[num], equal_nan=True)
                and (a.drop(columns=num).astype(str).values == b.drop(columns=num).astype(str).values).all())


def check(test, name, ok, detail=""):
    results.append({"test": test, "check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})
    print(f"{'PASS' if ok else 'FAIL'}  {test} | {name}  {detail}")

## A. Incremental – only in mode `incremental`
Removes the newest deltadate from the bucket and empties the DWH, loads without it, then runs the whole pipeline.

In [ ]:
if MODE == "incremental":
    keys = list_keys("weather/")
    last = max(deltadate_of(k) for k in keys)
    removed = {k: read_csv(k) for k in keys if deltadate_of(k) == last}
    removed_rows = sum(len(df) for df in removed.values())
    print(f"removing weather/deltadate={last}/ ({len(removed)} files, {removed_rows} rows) and emptying the DWH")

    s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": [{"Key": k} for k in removed]})
    conn.execute("TRUNCATE dwh.fact_weather_daily, dwh.dim_city, dwh.dim_date, dwh.dim_weather_type RESTART IDENTITY")
    conn.execute("TRUNCATE stg.weather_daily, stg.cities, stg.weather_types")
    check("A", "bucket without the deltadate", not any(deltadate_of(k) == last for k in list_keys("weather/")), last)
    check("A", "DWH empty", state()["fact_rows"] == 0)

    run("Load_dwh", "a1_initial")
    s = state()
    check("A", "initial load stops before the removed deltadate", s["max_deltadate"] < last, s["max_deltadate"])
    check("A", "initial load = all rows of the remaining files",
          s["fact_rows"] == value("SELECT count(*) FROM stg.weather_daily"), s["fact_rows"])
    initial_rows = s["fact_rows"]

    run("Ingest_data", "a2_pipeline")
    run("Load_dwh", "a2_pipeline")
    s = state()
    bucket_keys = set(list_keys("weather/"))
    check("A", "ingest wrote the removed files again", all(k in bucket_keys for k in removed), f"{len(removed)} files")
    check("A", "re-fetched files have the same values", all(same_values(read_csv(k), df) for k, df in removed.items()))
    check("A", "removed deltadate loaded again",
          value(f"SELECT count(*) FROM dwh.fact_weather_daily WHERE deltadate = '{last}'") == removed_rows, removed_rows)
    check("A", "only the new rows were added",
          s["fact_rows"] == initial_rows + value("SELECT count(*) FROM stg.weather_daily"),
          f"{initial_rows} -> {s['fact_rows']}")
else:
    print("skipped (mode upsert)")

## C. Idempotence – catch up once, then a second run must change nothing

In [ ]:
run("Ingest_data", "c1_catch_up")
run("Load_dwh", "c1_catch_up")
before, keys_before = state(), list_keys("")

run("Ingest_data", "c2_repeat")
run("Load_dwh", "c2_repeat")
after, keys_after = state(), list_keys("")

check("C", "no new files in the bucket", keys_after == keys_before, f"{len(keys_after)} files")
check("C", "DWH unchanged", after == before, after)
check("C", "second load staged nothing", value("SELECT count(*) FROM stg.weather_daily") == 0)

## B. Upsert – in a transaction that is rolled back
Stages the newest deltadate again (rows that are already in the fact) and runs the procedures.

In [ ]:
latest = state()["max_deltadate"]
staged = pd.concat(read_csv(k) for k in list_keys("weather/") if deltadate_of(k) == latest)
fact_rows = state()["fact_rows"]
city = staged["city"].min()
original = fact_value(city, latest)
print(f"re-staging deltadate {latest}: {len(staged)} rows")

with conn.transaction() as tx:
    stage("weather_daily", staged)
    conn.execute("CALL dwh.load_fact_weather_daily()")
    check("B", "procedure upserted every staged row", last_notice_rows() == len(staged), notices[-1])
    check("B", "row count unchanged (updated, not inserted)", state()["fact_rows"] == fact_rows, fact_rows)
    check("B", "no duplicate (date, city)", value("""SELECT count(*) FROM (SELECT 1 FROM dwh.fact_weather_daily
                                                   GROUP BY date_key, city_key HAVING count(*) > 1) d""") == 0)
    check("B", "updated rows got a new loaded_at",
          value("SELECT count(*) FROM dwh.fact_weather_daily WHERE loaded_at = now()") == len(staged))

    conn.execute(f"UPDATE stg.weather_daily SET temperature_2m_max = 99 WHERE city = '{city}'")
    conn.execute("CALL dwh.load_fact_weather_daily()")
    changed = fact_value(city, latest)
    check("B", "changed value arrives in the fact", changed == 99, f"{city}: {original} -> {changed}")

    stage("cities", pd.concat(read_csv(k) for k in list_keys("cities/")))
    conn.execute("CALL dwh.load_dim_city()")
    check("B", "dim_city: unchanged cities are not touched", last_notice_rows() == 0, notices[-1])
    key_before = value(f"SELECT city_key FROM dwh.dim_city WHERE city = '{city}'")
    conn.execute(f"UPDATE stg.cities SET population = population + 1 WHERE city = '{city}'")
    conn.execute("CALL dwh.load_dim_city()")
    check("B", "dim_city SCD1: one row updated", last_notice_rows() == 1, notices[-1])
    check("B", "dim_city SCD1: same city_key",
          value(f"SELECT city_key FROM dwh.dim_city WHERE city = '{city}'") == key_before, f"{city}: key {key_before}")
    raise psycopg.Rollback(tx)

check("B", "rollback: data unchanged", fact_value(city, latest) == original and state()["fact_rows"] == fact_rows,
      f"{city}: {fact_value(city, latest)}")

## Checks – `sql/checks/dwh_checks.sql`

In [ ]:
sql = (ROOT / "sql" / "checks" / "dwh_checks.sql").read_text(encoding="utf-8")
statements = [s.strip().rstrip(";") for s in re.split(r";\s*\n", sql)
              if any(line.strip() and not line.startswith("--") for line in s.splitlines())]
missing_days, gaps, problems = (query(statements[i]) for i in (2, 3, 4))
check("checks", "every city has every day", missing_days.empty, f"{len(missing_days)} cities short")
check("checks", "no gaps in dim_date", gaps.empty)
check("checks", "no orphans, unknown codes or wrong deltadates", int(problems.iloc[0].sum()) == 0,
      problems.iloc[0].to_dict())
query(statements[1])

## Result

In [ ]:
report = pd.DataFrame(results)
failed = report[report["result"] == "FAIL"]
print(f"{len(report) - len(failed)} passed, {len(failed)} failed")
if len(failed):
    raise AssertionError("failed checks:\n" + failed.to_string(index=False))
report